<a href="https://colab.research.google.com/github/nicopicomoco-sekihan-80/Antencoder-AntVLA/blob/main/Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# PGLA Milestone 1
# Physics-Grounded Latent Action
# ============================================

import os
import math
import random
import json
import time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [ ]:
# ============================================
# Configuration
# ============================================

VISION_DIM = 512
ACTION_DIM = 7
LATENT_DIM = 128

EMBED_DIM = 96
HIDDEN_DIM = 192

VOCAB_SIZE = 1000
MAX_LEN = 12

TRAIN_SIZE = 30000
VAL_SIZE = 5000
TEST_SIZE = 5000

BATCH_SIZE = 256

TEACHER_EPOCHS = 20
STUDENT_EPOCHS = 20

LR_TEACHER = 3e-4
LR_STUDENT = 3e-4

print("Configuration ready.")


Configuration ready.


In [ ]:
# ============================================
# Synthetic Language
# ============================================

SPECIAL_TOKENS = {
    "<PAD>": 0,
    "<UNK>": 1,
}

COLORS = ["red", "blue", "green"]
OBJECT_WORD = "cube"

ACTION_WORDS = [
    "pick",
    "push",
    "pull",
    "place",
]

DIRECTION_WORDS = [
    "left",
    "right",
    "forward",
    "backward",
]

VOCAB_WORDS = (
    list(SPECIAL_TOKENS.keys())
    + ACTION_WORDS
    + COLORS
    + [OBJECT_WORD]
    + DIRECTION_WORDS
    + [
        "up",
        "the",
        "to",
        "on",
    ]
)

word2id = {
    word: i
    for i, word in enumerate(VOCAB_WORDS)
}

id2word = {
    i: word
    for word, i in word2id.items()
}

VOCAB_SIZE = max(VOCAB_SIZE, len(word2id))

print("Vocabulary size:", len(word2id))
print(word2id)


Vocabulary size: 18
{'<PAD>': 0, '<UNK>': 1, 'pick': 2, 'push': 3, 'pull': 4, 'place': 5, 'red': 6, 'blue': 7, 'green': 8, 'cube': 9, 'left': 10, 'right': 11, 'forward': 12, 'backward': 13, 'up': 14, 'the': 15, 'to': 16, 'on': 17}


In [ ]:
# ============================================
# Tiny Language Encoder
# ============================================

class TinyLanguageEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=96,
        hidden_dim=192,
        latent_dim=128,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0,
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.projector = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.GELU(),
            nn.Linear(256, latent_dim),
            nn.Tanh(),
        )

    def forward(self, tokens, mask=None):

        x = self.embedding(tokens)

        h, _ = self.gru(x)

        if mask is None:
            pooled = h.mean(dim=1)

        else:
            mask = mask.unsqueeze(-1).float()

            pooled = (
                h * mask
            ).sum(dim=1)

            pooled = pooled / mask.sum(
                dim=1
            ).clamp_min(1.0)

        return self.projector(pooled)


In [ ]:
# ============================================
# Physical Action Generator
# ============================================

def make_action(scene, command):

    action_type = command["action"]
    color = command["color"]

    obj = scene[color].copy()

    # ----------------------------------------
    # PICK
    # ----------------------------------------
    if action_type == "pick":

        # approach / descend / grasp
        x = obj[0]
        y = obj[1]
        z = obj[2]

        roll = 0.0
        pitch = 0.0
        yaw = 0.0

        gripper = 1.0

        action = np.array([
            x,
            y,
            z,
            roll,
            pitch,
            yaw,
            gripper,
        ])

    # ----------------------------------------
    # PUSH
    # ----------------------------------------
    elif action_type == "push":

        direction = command["direction"]

        delta = {
            "left":     np.array([-0.20, 0.0, 0.0]),
            "right":    np.array([ 0.20, 0.0, 0.0]),
            "forward":  np.array([0.0,  0.20, 0.0]),
            "backward": np.array([0.0, -0.20, 0.0]),
        }[direction]

        target = obj + delta

        action = np.array([
            target[0],
            target[1],
            target[2],
            0.0,
            0.0,
            0.0,
            0.0,
        ])

    # ----------------------------------------
    # PULL
    # ----------------------------------------
    elif action_type == "pull":

        direction = command["direction"]

        delta = {
            "left":     np.array([-0.20, 0.0, 0.0]),
            "right":    np.array([ 0.20, 0.0, 0.0]),
            "forward":  np.array([0.0,  0.20, 0.0]),
            "backward": np.array([0.0, -0.20, 0.0]),
        }[direction]

        target = obj - delta

        action = np.array([
            target[0],
            target[1],
            target[2],
            0.0,
            0.0,
            0.0,
            0.0,
        ])

    # ----------------------------------------
    # PLACE
    # ----------------------------------------
    elif action_type == "place":

        target = np.array([
            np.random.uniform(-0.7, 0.7),
            np.random.uniform(-0.7, 0.7),
            0.05,
        ])

        action = np.array([
            target[0],
            target[1],
            target[2],
            0.0,
            0.0,
            0.0,
            0.0,
        ])

    else:
        raise ValueError(action_type)

    return action.astype(np.float32)


In [ ]:
# ============================================
# Command Generator
# ============================================

def make_command():

    action_type = random.choice(
        ACTION_WORDS
    )

    color = random.choice(
        COLORS
    )

    command = {
        "action": action_type,
        "color": color,
    }

    if action_type in ["push", "pull"]:
        command["direction"] = random.choice(
            DIRECTION_WORDS
        )

    return command


def command_to_text(command):

    action = command["action"]
    color = command["color"]

    if action == "pick":
        return [
            "pick",
            "up",
            "the",
            color,
            "cube",
        ]

    if action == "push":
        return [
            "push",
            "the",
            color,
            "cube",
            command["direction"],
        ]

    if action == "pull":
        return [
            "pull",
            "the",
            color,
            "cube",
            command["direction"],
        ]

    if action == "place":
        return [
            "place",
            "the",
            color,
            "cube",
        ]

    raise ValueError(action)


In [ ]:
# ============================================
# Synthetic Vision Feature
# ============================================

SCENE_DIM = len(COLORS) * 3

vision_projection = torch.randn(
    SCENE_DIM,
    VISION_DIM,
)

vision_projection = (
    vision_projection
    / vision_projection.norm(
        dim=0,
        keepdim=True
    ).clamp_min(1e-8)
)


def scene_to_vector(scene):

    values = []

    for color in COLORS:
        values.extend(
            scene[color]
        )

    return np.asarray(
        values,
        dtype=np.float32
    )


def make_vision_feature(scene):

    state = torch.tensor(
        scene_to_vector(scene)
    )

    feat = state @ vision_projection

    feat += 0.01 * torch.randn(
        VISION_DIM
    )

    return feat.float()


In [ ]:
# ============================================
# Dataset
# ============================================

def tokenize(words):

    ids = [
        word2id.get(
            word,
            word2id["<UNK>"]
        )
        for word in words
    ]

    ids = ids[:MAX_LEN]

    mask = [1] * len(ids)

    while len(ids) < MAX_LEN:
        ids.append(
            word2id["<PAD>"]
        )
        mask.append(0)

    return (
        torch.tensor(ids).long(),
        torch.tensor(mask).long(),
    )


class PGLADataset(Dataset):

    def __init__(self, size):

        self.samples = []

        for _ in range(size):

            scene = make_scene()

            command = make_command()

            action = make_action(
                scene,
                command
            )

            text = command_to_text(
                command
            )

            tokens, mask = tokenize(
                text
            )

            vision = make_vision_feature(
                scene
            )

            self.samples.append({
                "vision": vision,
                "action": torch.tensor(
                    action
                ),
                "tokens": tokens,
                "mask": mask,
                "action_type":
                    command["action"],
                "color":
                    command["color"],
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


In [ ]:
# ============================================
# PGLA Milestone 1
# Synthetic World + Dataset
# ============================================

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------
# Constants
# --------------------------------------------

COLORS = ["red", "blue", "green"]

ACTION_WORDS = [
    "pick",
    "push",
    "pull",
    "place",
]

DIRECTION_WORDS = [
    "left",
    "right",
    "forward",
    "backward",
]

OBJECT_WORD = "cube"

MAX_LEN = 12


# --------------------------------------------
# Scene
# --------------------------------------------

def random_position():

    return np.random.uniform(
        low=[-0.8, -0.8, 0.05],
        high=[0.8, 0.8, 0.5],
    ).astype(np.float32)


def make_scene():

    objects = {}

    for color in COLORS:
        objects[color] = random_position()

    return objects


# --------------------------------------------
# Command
# --------------------------------------------

def make_command():

    action_type = np.random.choice(
        ACTION_WORDS
    )

    color = np.random.choice(
        COLORS
    )

    command = {
        "action": action_type,
        "color": color,
    }

    if action_type in ["push", "pull"]:

        command["direction"] = np.random.choice(
            DIRECTION_WORDS
        )

    return command


def command_to_text(command):

    action = command["action"]
    color = command["color"]

    if action == "pick":

        return [
            "pick",
            "up",
            "the",
            color,
            "cube",
        ]

    elif action == "push":

        return [
            "push",
            "the",
            color,
            "cube",
            command["direction"],
        ]

    elif action == "pull":

        return [
            "pull",
            "the",
            color,
            "cube",
            command["direction"],
        ]

    elif action == "place":

        return [
            "place",
            "the",
            color,
            "cube",
        ]

    raise ValueError(action)


# --------------------------------------------
# Action
# --------------------------------------------

def make_action(scene, command):

    action_type = command["action"]
    color = command["color"]

    obj = scene[color].copy()

    if action_type == "pick":

        action = np.array([
            obj[0],
            obj[1],
            obj[2],
            0.0,
            0.0,
            0.0,
            1.0,
        ])

    elif action_type == "push":

        direction = command["direction"]

        delta = {
            "left":
                np.array([-0.20, 0.0, 0.0]),

            "right":
                np.array([0.20, 0.0, 0.0]),

            "forward":
                np.array([0.0, 0.20, 0.0]),

            "backward":
                np.array([0.0, -0.20, 0.0]),
        }[direction]

        target = obj + delta

        action = np.array([
            target[0],
            target[1],
            target[2],
            0.0,
            0.0,
            0.0,
            0.0,
        ])

    elif action_type == "pull":

        direction = command["direction"]

        delta = {
            "left":
                np.array([-0.20, 0.0, 0.0]),

            "right":
                np.array([0.20, 0.0, 0.0]),

            "forward":
                np.array([0.0, 0.20, 0.0]),

            "backward":
                np.array([0.0, -0.20, 0.0]),
        }[direction]

        target = obj - delta

        action = np.array([
            target[0],
            target[1],
            target[2],
            0.0,
            0.0,
            0.0,
            0.0,
        ])

    elif action_type == "place":

        target = np.array([
            np.random.uniform(-0.7, 0.7),
            np.random.uniform(-0.7, 0.7),
            0.05,
        ])

        action = np.array([
            target[0],
            target[1],
            target[2],
            0.0,
            0.0,
            0.0,
            0.0,
        ])

    else:
        raise ValueError(action_type)

    return action.astype(np.float32)


# --------------------------------------------
# Vision feature
# --------------------------------------------

SCENE_DIM = len(COLORS) * 3

vision_projection = torch.randn(
    SCENE_DIM,
    VISION_DIM,
)

vision_projection = (
    vision_projection
    /
    vision_projection.norm(
        dim=0,
        keepdim=True
    ).clamp_min(1e-8)
)


def scene_to_vector(scene):

    values = []

    for color in COLORS:
        values.extend(
            scene[color]
        )

    return np.asarray(
        values,
        dtype=np.float32
    )


def make_vision_feature(scene):

    state = torch.tensor(
        scene_to_vector(scene)
    )

    feat = state @ vision_projection

    feat += (
        0.01
        * torch.randn(VISION_DIM)
    )

    return feat.float()


# --------------------------------------------
# Tokenizer
# --------------------------------------------

def tokenize(words):

    ids = [
        word2id.get(
            word,
            word2id["<UNK>"]
        )
        for word in words
    ]

    ids = ids[:MAX_LEN]

    mask = [1] * len(ids)

    while len(ids) < MAX_LEN:

        ids.append(
            word2id["<PAD>"]
        )

        mask.append(0)

    return (
        torch.tensor(ids).long(),
        torch.tensor(mask).long(),
    )


# --------------------------------------------
# Dataset
# --------------------------------------------

class PGLADataset(Dataset):

    def __init__(self, size):

        self.samples = []

        for _ in range(size):

            scene = make_scene()

            command = make_command()

            action = make_action(
                scene,
                command
            )

            text = command_to_text(
                command
            )

            tokens, mask = tokenize(
                text
            )

            vision = make_vision_feature(
                scene
            )

            self.samples.append({

                "vision":
                    vision,

                "action":
                    torch.tensor(action),

                "tokens":
                    tokens,

                "mask":
                    mask,

                "action_type":
                    command["action"],

                "color":
                    command["color"],
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


print("Synthetic world loaded.")
print("make_scene:", make_scene)
print("make_action:", make_action)
print("PGLADataset:", PGLADataset)


Synthetic world loaded.
make_scene: <function make_scene at 0x7ba4b99fe980>
make_action: <function make_action at 0x7ba4e35860c0>
PGLADataset: <class '__main__.PGLADataset'>


In [ ]:
test_dataset = PGLADataset(100)

sample = test_dataset[0]

print("Vision:", sample["vision"].shape)
print("Action:", sample["action"].shape)
print("Tokens:", sample["tokens"])
print("Mask:", sample["mask"])
print("Action type:", sample["action_type"])
print("Color:", sample["color"])


Vision: torch.Size([512])
Action: torch.Size([7])
Tokens: tensor([ 5, 15,  8,  9,  0,  0,  0,  0,  0,  0,  0,  0])
Mask: tensor([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0])
Action type: place
Color: green


In [ ]:
# ============================================
# Inspect several samples
# ============================================

for i in range(10):

    s = test_dataset[i]

    token_ids = s["tokens"]
    words = []

    for token_id, mask in zip(
        token_ids.tolist(),
        s["mask"].tolist()
    ):
        if mask == 1:
            words.append(
                id2word[token_id]
            )

    print(
        f"{i:02d} | "
        f"{' '.join(words):35s} | "
        f"{s['action_type']:5s} | "
        f"{s['color']:5s} | "
        f"action={s['action'].numpy()}"
    )


00 | place the green cube                | place | green | action=[-0.6711817  0.6578738  0.05       0.         0.         0.
  0.       ]
01 | push the green cube right           | push  | green | action=[ 0.09111203 -0.33403337  0.3253338   0.          0.          0.
  0.        ]
02 | pick up the blue cube               | pick  | blue  | action=[ 0.7731694  -0.05317937  0.43697318  0.          0.          0.
  1.        ]
03 | place the blue cube                 | place | blue  | action=[ 0.2275312 -0.2636045  0.05       0.         0.         0.
  0.       ]
04 | place the blue cube                 | place | blue  | action=[-0.42562398 -0.6366818   0.05        0.          0.          0.
  0.        ]
05 | pick up the red cube                | pick  | red   | action=[-0.27947146 -0.17811634  0.17210707  0.          0.          0.
  1.        ]
06 | pull the blue cube left             | pull  | blue  | action=[-0.59116465  0.5047383   0.3680858   0.          0.          0.
  0.       

In [ ]:
TRAIN_SIZE = 30000
VAL_SIZE = 5000
TEST_SIZE = 5000

train_dataset = PGLADataset(TRAIN_SIZE)
val_dataset = PGLADataset(VAL_SIZE)
test_dataset = PGLADataset(TEST_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


Train: 30000
Val: 5000
Test: 5000


In [ ]:
# ============================================
# PGLA Teacher
# V + A -> z -> A
# ============================================

class PhysicalTeacher(nn.Module):

    def __init__(
        self,
        vision_dim=512,
        action_dim=7,
        latent_dim=128,
    ):
        super().__init__()

        self.vision_encoder = nn.Sequential(
            nn.Linear(vision_dim, 256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.GELU(),
        )

        self.action_encoder = nn.Sequential(
            nn.Linear(action_dim, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
        )

        self.fusion = nn.Sequential(
            nn.Linear(256, 256),
            nn.GELU(),
            nn.Linear(256, 256),
            nn.GELU(),
        )

        self.bottleneck = nn.Sequential(
            nn.Linear(256, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.Tanh(),
        )

        self.action_head = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.GELU(),
            nn.Linear(128, action_dim),
        )

    def forward(self, vision, action):

        v = self.vision_encoder(vision)

        a = self.action_encoder(action)

        fused = self.fusion(
            torch.cat([v, a], dim=-1)
        )

        z = self.bottleneck(fused)

        action_pred = self.action_head(z)

        return {
            "latent": z,
            "action": action_pred,
        }


teacher = PhysicalTeacher(
    vision_dim=VISION_DIM,
    action_dim=ACTION_DIM,
    latent_dim=LATENT_DIM,
).to(device)

print(
    "Teacher parameters:",
    sum(p.numel() for p in teacher.parameters())
)


Teacher parameters: 363911


In [ ]:
# ============================================
# Action Loss
# ============================================

def action_loss_components(pred, target):

    position_loss = F.smooth_l1_loss(
        pred[:, :3],
        target[:, :3]
    )

    rotation_loss = F.smooth_l1_loss(
        pred[:, 3:6],
        target[:, 3:6]
    )

    # Datasetでは gripper:
    # 1 = closed
    # 0 = open
    #
    # BCEWithLogitsなので
    # predはlogitのまま使用

    gripper_loss = F.binary_cross_entropy_with_logits(
        pred[:, 6],
        target[:, 6]
    )

    total = (
        position_loss
        + rotation_loss
        + 0.5 * gripper_loss
    )

    return {
        "total": total,
        "position": position_loss,
        "rotation": rotation_loss,
        "gripper": gripper_loss,
    }


In [ ]:
# ============================================
# Train Teacher
# ============================================

optimizer = torch.optim.AdamW(
    teacher.parameters(),
    lr=LR_TEACHER,
    weight_decay=1e-4,
)

teacher_history = []

for epoch in range(TEACHER_EPOCHS):

    teacher.train()

    running = {
        "total": 0.0,
        "position": 0.0,
        "rotation": 0.0,
        "gripper": 0.0,
    }

    n_samples = 0

    for batch in train_loader:

        vision = batch["vision"].to(
            device,
            non_blocking=True
        )

        action = batch["action"].to(
            device,
            non_blocking=True
        )

        output = teacher(
            vision,
            action
        )

        losses = action_loss_components(
            output["action"],
            action
        )

        loss = losses["total"]

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        bs = vision.size(0)

        n_samples += bs

        for key in running:
            running[key] += (
                losses[key].item() * bs
            )

    epoch_stats = {
        key: running[key] / n_samples
        for key in running
    }

    teacher_history.append(
        epoch_stats
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Total {epoch_stats['total']:.6f} | "
        f"Pos {epoch_stats['position']:.6f} | "
        f"Rot {epoch_stats['rotation']:.6f} | "
        f"Grip {epoch_stats['gripper']:.6f}"
    )


Epoch 01 | Total 0.114955 | Pos 0.017186 | Rot 0.000768 | Grip 0.194001
Epoch 02 | Total 0.002857 | Pos 0.001517 | Rot 0.000030 | Grip 0.002620
Epoch 03 | Total 0.000593 | Pos 0.000100 | Rot 0.000016 | Grip 0.000954
Epoch 04 | Total 0.000281 | Pos 0.000025 | Rot 0.000008 | Grip 0.000497
Epoch 05 | Total 0.000175 | Pos 0.000014 | Rot 0.000005 | Grip 0.000312
Epoch 06 | Total 0.000121 | Pos 0.000010 | Rot 0.000004 | Grip 0.000215
Epoch 07 | Total 0.000090 | Pos 0.000008 | Rot 0.000003 | Grip 0.000158
Epoch 08 | Total 0.000070 | Pos 0.000007 | Rot 0.000002 | Grip 0.000121
Epoch 09 | Total 0.000059 | Pos 0.000009 | Rot 0.000002 | Grip 0.000096
Epoch 10 | Total 0.000050 | Pos 0.000009 | Rot 0.000002 | Grip 0.000077
Epoch 11 | Total 0.000047 | Pos 0.000013 | Rot 0.000002 | Grip 0.000064
Epoch 12 | Total 0.000039 | Pos 0.000011 | Rot 0.000001 | Grip 0.000054
Epoch 13 | Total 0.000037 | Pos 0.000012 | Rot 0.000002 | Grip 0.000046
Epoch 14 | Total 0.000035 | Pos 0.000012 | Rot 0.000003 | Grip 0

In [ ]:
# ============================================
# Teacher Evaluation - FIXED
# ============================================

@torch.no_grad()
def evaluate_teacher(model, loader):

    model.eval()

    total_abs = []
    position_abs = []
    rotation_abs = []
    gripper_correct = []

    for batch in loader:

        vision = batch["vision"].to(device)
        action = batch["action"].to(device)

        output = model(
            vision,
            action
        )

        pred = output["action"]

        # ----------------------------------------
        # Overall action MAE
        # ----------------------------------------

        total_abs.extend(
            (pred - action)
            .abs()
            .mean(dim=1)
            .cpu()
            .numpy()
        )

        # ----------------------------------------
        # Position MAE
        # ----------------------------------------

        position_abs.extend(
            (pred[:, :3] - action[:, :3])
            .abs()
            .mean(dim=1)
            .cpu()
            .numpy()
        )

        # ----------------------------------------
        # Rotation MAE
        # ----------------------------------------

        rotation_abs.extend(
            (pred[:, 3:6] - action[:, 3:6])
            .abs()
            .mean(dim=1)
            .cpu()
            .numpy()
        )

        # ----------------------------------------
        # Gripper accuracy
        # ----------------------------------------

        grip_pred = (
            torch.sigmoid(
                pred[:, 6]
            ) > 0.5
        ).float()

        gripper_correct.extend(
            (
                grip_pred
                ==
                action[:, 6]
            )
            .float()
            .cpu()
            .numpy()
        )

    return {
        "action_mae":
            float(np.mean(total_abs)),

        "position_mae":
            float(np.mean(position_abs)),

        "rotation_mae":
            float(np.mean(rotation_abs)),

        "gripper_accuracy":
            float(np.mean(gripper_correct)),
    }


teacher_results = evaluate_teacher(
    teacher,
    val_loader
)

print(
    json.dumps(
        teacher_results,
        indent=2
    )
)


{
  "action_mae": 1.533691167831421,
  "position_mae": 0.001982106827199459,
  "rotation_mae": 0.0008923730929382145,
  "gripper_accuracy": 1.0
}


In [ ]:
# ============================================
# PGLA Student
# Language -> z -> Action
# ============================================

class TinyLanguageEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=96,
        hidden_dim=192,
        latent_dim=128,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0,
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.projector = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                256
            ),
            nn.GELU(),
            nn.Linear(
                256,
                latent_dim
            ),
            nn.Tanh(),
        )

    def forward(
        self,
        tokens,
        mask=None
    ):

        x = self.embedding(tokens)

        h, _ = self.gru(x)

        if mask is None:

            pooled = h.mean(dim=1)

        else:

            mask = mask.unsqueeze(-1).float()

            pooled = (
                h * mask
            ).sum(dim=1)

            pooled = pooled / (
                mask.sum(dim=1)
                .clamp_min(1.0)
            )

        return self.projector(pooled)


class PGLAStudent(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=96,
        hidden_dim=192,
        latent_dim=128,
        action_dim=7,
    ):
        super().__init__()

        self.language_encoder = TinyLanguageEncoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
        )

        self.action_head = nn.Sequential(
            nn.Linear(
                latent_dim,
                128
            ),
            nn.GELU(),
            nn.Linear(
                128,
                action_dim
            ),
        )

    def forward(
        self,
        tokens,
        mask
    ):

        z = self.language_encoder(
            tokens,
            mask
        )

        action = self.action_head(z)

        return {
            "latent": z,
            "action": action,
        }


student = PGLAStudent(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    action_dim=ACTION_DIM,
).to(device)

print(
    "Student parameters:",
    sum(
        p.numel()
        for p in student.parameters()
    )
)


Student parameters: 578951


In [ ]:
# ============================================
# Train Student
# ============================================

teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False


optimizer_student = torch.optim.AdamW(
    student.parameters(),
    lr=LR_STUDENT,
    weight_decay=1e-4,
)

LAMBDA_LATENT = 1.0
LAMBDA_ACTION = 0.5

student_history = []


for epoch in range(
    STUDENT_EPOCHS
):

    student.train()

    running = {
        "total": 0.0,
        "latent": 0.0,
        "action": 0.0,
    }

    n_samples = 0

    for batch in train_loader:

        vision = batch["vision"].to(
            device,
            non_blocking=True
        )

        action = batch["action"].to(
            device,
            non_blocking=True
        )

        tokens = batch["tokens"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        # ------------------------------------
        # Teacher
        # ------------------------------------

        with torch.no_grad():

            teacher_output = teacher(
                vision,
                action
            )

            z_teacher = (
                teacher_output["latent"]
            )

        # ------------------------------------
        # Student
        # ------------------------------------

        student_output = student(
            tokens,
            mask
        )

        z_student = (
            student_output["latent"]
        )

        action_pred = (
            student_output["action"]
        )

        # ------------------------------------
        # Loss
        # ------------------------------------

        latent_loss = F.mse_loss(
            z_student,
            z_teacher
        )

        action_losses = (
            action_loss_components(
                action_pred,
                action
            )
        )

        action_loss = (
            action_losses["total"]
        )

        loss = (
            LAMBDA_LATENT
            * latent_loss
            +
            LAMBDA_ACTION
            * action_loss
        )

        optimizer_student.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer_student.step()

        bs = tokens.size(0)

        n_samples += bs

        running["total"] += (
            loss.item() * bs
        )

        running["latent"] += (
            latent_loss.item() * bs
        )

        running["action"] += (
            action_loss.item() * bs
        )

    stats = {
        k: v / n_samples
        for k, v in running.items()
    }

    student_history.append(stats)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Total={stats['total']:.6f} | "
        f"Latent={stats['latent']:.6f} | "
        f"Action={stats['action']:.6f}"
    )


Epoch 01 | Total=0.204379 | Latent=0.121314 | Action=0.166131
Epoch 02 | Total=0.062993 | Latent=0.025661 | Action=0.074664
Epoch 03 | Total=0.062254 | Latent=0.025465 | Action=0.073580
Epoch 04 | Total=0.062098 | Latent=0.025387 | Action=0.073423
Epoch 05 | Total=0.061902 | Latent=0.025344 | Action=0.073116
Epoch 06 | Total=0.061775 | Latent=0.025308 | Action=0.072934
Epoch 07 | Total=0.061895 | Latent=0.025326 | Action=0.073138
Epoch 08 | Total=0.061970 | Latent=0.025358 | Action=0.073225
Epoch 09 | Total=0.061836 | Latent=0.025311 | Action=0.073049
Epoch 10 | Total=0.061783 | Latent=0.025308 | Action=0.072951
Epoch 11 | Total=0.061836 | Latent=0.025336 | Action=0.073000
Epoch 12 | Total=0.061777 | Latent=0.025330 | Action=0.072894
Epoch 13 | Total=0.061737 | Latent=0.025309 | Action=0.072854
Epoch 14 | Total=0.061683 | Latent=0.025305 | Action=0.072757
Epoch 15 | Total=0.061689 | Latent=0.025277 | Action=0.072824
Epoch 16 | Total=0.061247 | Latent=0.025093 | Action=0.072307
Epoch 17

In [ ]:
# ============================================
# Student Evaluation
# ============================================

@torch.no_grad()
def evaluate_student(
    model,
    loader,
    teacher=None
):

    model.eval()

    latent_mse_list = []
    cosine_list = []

    action_mae_list = []
    position_mae_list = []
    rotation_mae_list = []

    gripper_correct = []

    for batch in loader:

        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        action = batch["action"].to(device)

        output = model(
            tokens,
            mask
        )

        z_student = output["latent"]
        pred = output["action"]

        # ------------------------------------
        # Action
        # ------------------------------------

        action_mae_list.extend(
            (
                pred - action
            )
            .abs()
            .mean(dim=1)
            .cpu()
            .numpy()
        )

        position_mae_list.extend(
            (
                pred[:, :3]
                -
                action[:, :3]
            )
            .abs()
            .mean(dim=1)
            .cpu()
            .numpy()
        )

        rotation_mae_list.extend(
            (
                pred[:, 3:6]
                -
                action[:, 3:6]
            )
            .abs()
            .mean(dim=1)
            .cpu()
            .numpy()
        )

        grip_pred = (
            torch.sigmoid(
                pred[:, 6]
            )
            > 0.5
        ).float()

        gripper_correct.extend(
            (
                grip_pred
                ==
                action[:, 6]
            )
            .float()
            .cpu()
            .numpy()
        )

        # ------------------------------------
        # Latent
        # ------------------------------------

        if teacher is not None:

            vision = batch["vision"].to(
                device
            )

            teacher_output = teacher(
                vision,
                action
            )

            z_teacher = (
                teacher_output["latent"]
            )

            latent_mse_list.extend(
                (
                    z_student
                    -
                    z_teacher
                )
                .pow(2)
                .mean(dim=1)
                .cpu()
                .numpy()
            )

            cosine_list.extend(
                F.cosine_similarity(
                    z_student,
                    z_teacher,
                    dim=1
                )
                .cpu()
                .numpy()
            )

    return {
        "latent_mse":
            float(
                np.mean(latent_mse_list)
            ),

        "latent_cosine":
            float(
                np.mean(cosine_list)
            ),

        "action_mae":
            float(
                np.mean(action_mae_list)
            ),

        "position_mae":
            float(
                np.mean(position_mae_list)
            ),

        "rotation_mae":
            float(
                np.mean(rotation_mae_list)
            ),

        "gripper_accuracy":
            float(
                np.mean(gripper_correct)
            ),
    }


student_results = evaluate_student(
    student,
    test_loader,
    teacher
)

print(
    json.dumps(
        student_results,
        indent=2
    )
)


{
  "latent_mse": 0.024539856240153313,
  "latent_cosine": 0.976845920085907,
  "action_mae": 1.528059720993042,
  "position_mae": 0.29068994522094727,
  "rotation_mae": 0.003833048976957798,
  "gripper_accuracy": 1.0
}


In [ ]:
# ============================================
# Student latent collection
# ============================================

@torch.no_grad()
def collect_student_latents(model, loader):

    model.eval()

    latents = []
    action_types = []
    colors = []

    for batch in loader:

        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)

        output = model(
            tokens,
            mask
        )

        latents.append(
            output["latent"].cpu()
        )

        action_types.extend(
            batch["action_type"]
        )

        colors.extend(
            batch["color"]
        )

    return (
        torch.cat(latents),
        action_types,
        colors,
    )


z_student_test, student_types, student_colors = (
    collect_student_latents(
        student,
        test_loader
    )
)

print(
    "Student latent:",
    z_student_test.shape
)
# ============================================
# Latent centroid by action type
# ============================================

for action_type in ACTION_WORDS:

    indices = [
        i
        for i, t in enumerate(student_types)
        if t == action_type
    ]

    z = z_student_test[indices]

    centroid = z.mean(dim=0)

    print(
        f"{action_type:6s} "
        f"N={len(indices):4d} "
        f"centroid_norm={centroid.norm():.4f}"
    )


Student latent: torch.Size([5000, 128])
pick   N=1243 centroid_norm=7.9545
push   N=1225 centroid_norm=8.1144
pull   N=1243 centroid_norm=8.1370
place  N=1289 centroid_norm=8.2449


In [ ]:
# ============================================
# Latent Geometry Analysis
# ============================================

from itertools import combinations

# --------------------------------------------
# Centroids
# --------------------------------------------

centroids = {}

for action_type in ACTION_WORDS:

    indices = [
        i
        for i, t in enumerate(student_types)
        if t == action_type
    ]

    z = z_student_test[indices]

    centroids[action_type] = z.mean(dim=0)


# --------------------------------------------
# Between-class cosine similarity
# --------------------------------------------

print("\n=== Between-class cosine similarity ===")

for a, b in combinations(ACTION_WORDS, 2):

    sim = F.cosine_similarity(
        centroids[a].unsqueeze(0),
        centroids[b].unsqueeze(0),
    ).item()

    print(
        f"{a:6s} vs {b:6s}: "
        f"{sim:.4f}"
    )


# --------------------------------------------
# Within-class cosine similarity
# --------------------------------------------

print("\n=== Within-class cosine similarity ===")

for action_type in ACTION_WORDS:

    indices = [
        i
        for i, t in enumerate(student_types)
        if t == action_type
    ]

    z = z_student_test[indices]

    centroid = centroids[action_type]

    sim = F.cosine_similarity(
        z,
        centroid.unsqueeze(0),
        dim=1
    ).mean().item()

    print(
        f"{action_type:6s}: "
        f"{sim:.4f}"
    )



=== Between-class cosine similarity ===
pick   vs push  : -0.3808
pick   vs pull  : -0.3743
pick   vs place : -0.4001
push   vs pull  : 0.9996
push   vs place : 0.9940
pull   vs place : 0.9937

=== Within-class cosine similarity ===
pick  : 1.0000
push  : 0.9969
pull  : 0.9977
place : 1.0000


In [ ]:
# ============================================
# Linear Probe: Action Type
# ============================================

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = z_student_test.numpy()

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    student_types
)

# --------------------------------------------
# Train / test split
# --------------------------------------------

split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# --------------------------------------------
# Linear probe
# --------------------------------------------

probe = LogisticRegression(
    max_iter=2000,
    random_state=42
)

probe.fit(
    X_train,
    y_train
)

pred = probe.predict(
    X_test
)

accuracy = accuracy_score(
    y_test,
    pred
)

print(
    f"Action-type linear probe accuracy: "
    f"{accuracy:.4f}"
)


Action-type linear probe accuracy: 1.0000


In [ ]:
# ============================================
# Save PGLA Phase-1 baseline
# ============================================

torch.save({
    "teacher": teacher.state_dict(),
    "student": student.state_dict(),

    "config": {
        "vision_dim": VISION_DIM,
        "action_dim": ACTION_DIM,
        "latent_dim": LATENT_DIM,
        "vocab_size": VOCAB_SIZE,
    },

    "teacher_results": teacher_results,
    "student_results": student_results,

    "action_probe_accuracy": accuracy,
}, "pgla_phase1_baseline.pt")

print(
    "Saved: pgla_phase1_baseline.pt"
)


Saved: pgla_phase1_baseline.pt


In [ ]:
# ============================================
# PGLA Ablation: Latent Dimension Sweep
# ============================================

LATENT_DIMS = [32, 64, 128, 256, 512]

SWEEP_EPOCHS_TEACHER = 15
SWEEP_EPOCHS_STUDENT = 15

SWEEP_LR_TEACHER = 1e-3
SWEEP_LR_STUDENT = 1e-3

print("Latent dimensions:", LATENT_DIMS)


Latent dimensions: [32, 64, 128, 256, 512]


In [ ]:
# ============================================
# Model factory
# ============================================

def make_teacher(latent_dim):

    return PhysicalTeacher(
        vision_dim=VISION_DIM,
        action_dim=ACTION_DIM,
        latent_dim=latent_dim,
    ).to(device)


def make_student(latent_dim):

    return PGLAStudent(
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        latent_dim=latent_dim,
        action_dim=ACTION_DIM,
    ).to(device)


In [ ]:
# ============================================
# Train Teacher for one latent dimension
# ============================================

def train_teacher_model(
    latent_dim,
    epochs=15,
):

    model = make_teacher(
        latent_dim
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SWEEP_LR_TEACHER,
        weight_decay=1e-4,
    )

    for epoch in range(epochs):

        model.train()

        total_loss = 0.0
        n = 0

        for batch in train_loader:

            vision = batch["vision"].to(
                device,
                non_blocking=True
            )

            action = batch["action"].to(
                device,
                non_blocking=True
            )

            output = model(
                vision,
                action
            )

            losses = action_loss_components(
                output["action"],
                action
            )

            loss = losses["total"]

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            optimizer.step()

            bs = vision.size(0)

            total_loss += (
                loss.item() * bs
            )

            n += bs

    result = evaluate_teacher(
        model,
        val_loader
    )

    return model, result


In [ ]:
# ============================================
# Train Student for one latent dimension
# ============================================

def train_student_model(
    teacher_model,
    latent_dim,
    epochs=15,
):

    student_model = make_student(
        latent_dim
    )

    teacher_model.eval()

    for p in teacher_model.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(
        student_model.parameters(),
        lr=SWEEP_LR_STUDENT,
        weight_decay=1e-4,
    )

    for epoch in range(epochs):

        student_model.train()

        for batch in train_loader:

            vision = batch["vision"].to(
                device,
                non_blocking=True
            )

            action = batch["action"].to(
                device,
                non_blocking=True
            )

            tokens = batch["tokens"].to(
                device,
                non_blocking=True
            )

            mask = batch["mask"].to(
                device,
                non_blocking=True
            )

            # --------------------------------
            # Teacher target
            # --------------------------------

            with torch.no_grad():

                teacher_output = teacher_model(
                    vision,
                    action
                )

                z_teacher = (
                    teacher_output["latent"]
                )

            # --------------------------------
            # Student
            # --------------------------------

            student_output = student_model(
                tokens,
                mask
            )

            z_student = (
                student_output["latent"]
            )

            action_pred = (
                student_output["action"]
            )

            # --------------------------------
            # Loss
            # --------------------------------

            latent_loss = F.mse_loss(
                z_student,
                z_teacher
            )

            action_losses = (
                action_loss_components(
                    action_pred,
                    action
                )
            )

            action_loss = (
                action_losses["total"]
            )

            loss = (
                latent_loss
                +
                0.5 * action_loss
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            optimizer.step()

    return student_model


In [ ]:
# ============================================
# Action-type linear probe
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


def evaluate_action_probe(
    model,
    loader,
):

    z_list = []
    labels = []

    model.eval()

    with torch.no_grad():

        for batch in loader:

            tokens = batch["tokens"].to(
                device,
                non_blocking=True
            )

            mask = batch["mask"].to(
                device,
                non_blocking=True
            )

            output = model(
                tokens,
                mask
            )

            z_list.append(
                output["latent"].cpu()
            )

            labels.extend(
                batch["action_type"]
            )

    X = torch.cat(
        z_list,
        dim=0
    ).numpy()

    encoder = LabelEncoder()

    y = encoder.fit_transform(
        labels
    )

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
            stratify=y,
        )
    )

    probe = LogisticRegression(
        max_iter=2000,
        random_state=42,
    )

    probe.fit(
        X_train,
        y_train,
    )

    pred = probe.predict(
        X_test
    )

    return accuracy_score(
        y_test,
        pred
    )


In [ ]:
# ============================================
# RUN LATENT DIMENSION SWEEP
# ============================================

sweep_results = []

for latent_dim in LATENT_DIMS:

    print("\n")
    print("=" * 60)
    print(
        f"LATENT DIM = {latent_dim}"
    )
    print("=" * 60)

    # ----------------------------------------
    # Teacher
    # ----------------------------------------

    teacher_model, teacher_result = (
        train_teacher_model(
            latent_dim,
            epochs=SWEEP_EPOCHS_TEACHER,
        )
    )

    print(
        "Teacher position MAE:",
        teacher_result["position_mae"]
    )

    # ----------------------------------------
    # Student
    # ----------------------------------------

    student_model = train_student_model(
        teacher_model,
        latent_dim,
        epochs=SWEEP_EPOCHS_STUDENT,
    )

    # ----------------------------------------
    # Student evaluation
    # ----------------------------------------

    student_result = evaluate_student(
        student_model,
        test_loader,
        teacher_model,
    )

    # ----------------------------------------
    # Linear probe
    # ----------------------------------------

    probe_accuracy = (
        evaluate_action_probe(
            student_model,
            test_loader,
        )
    )

    # ----------------------------------------
    # Parameter count
    # ----------------------------------------

    params = sum(
        p.numel()
        for p in student_model.parameters()
    )

    result = {

        "latent_dim":
            latent_dim,

        "student_params":
            params,

        "teacher_position_mae":
            teacher_result[
                "position_mae"
            ],

        "student_position_mae":
            student_result[
                "position_mae"
            ],

        "student_rotation_mae":
            student_result[
                "rotation_mae"
            ],

        "student_gripper_accuracy":
            student_result[
                "gripper_accuracy"
            ],

        "latent_mse":
            student_result[
                "latent_mse"
            ],

        "latent_cosine":
            student_result[
                "latent_cosine"
            ],

        "action_probe_accuracy":
            probe_accuracy,
    }

    sweep_results.append(
        result
    )

    print(
        json.dumps(
            result,
            indent=2
        )
    )

    # ----------------------------------------
    # Free GPU memory
    # ----------------------------------------

    del teacher_model
    del student_model

    torch.cuda.empty_cache()




LATENT DIM = 32
Teacher position MAE: 0.008932622149586678
{
  "latent_dim": 32,
  "student_params": 541991,
  "teacher_position_mae": 0.008932622149586678,
  "student_position_mae": 0.2910529673099518,
  "student_rotation_mae": 0.0012218225747346878,
  "student_gripper_accuracy": 1.0,
  "latent_mse": 0.02818756364285946,
  "latent_cosine": 0.9751179814338684,
  "action_probe_accuracy": 1.0
}


LATENT DIM = 64
Teacher position MAE: 0.0064482237212359905
{
  "latent_dim": 64,
  "student_params": 554311,
  "teacher_position_mae": 0.0064482237212359905,
  "student_position_mae": 0.28976014256477356,
  "student_rotation_mae": 0.004345814697444439,
  "student_gripper_accuracy": 1.0,
  "latent_mse": 0.030511630699038506,
  "latent_cosine": 0.9714553952217102,
  "action_probe_accuracy": 1.0
}


LATENT DIM = 128
Teacher position MAE: 0.010234817862510681
{
  "latent_dim": 128,
  "student_params": 578951,
  "teacher_position_mae": 0.010234817862510681,
  "student_position_mae": 0.289936780929

In [ ]:
# ============================================
# PGLA Vision-Conditioned Action Model
# Existing Student + Vision + Action Head
# ============================================

class PGLAVisionAction(nn.Module):

    def __init__(
        self,
        student,
        vision_dim=512,
        latent_dim=128,
        action_dim=7,
        hidden_dim=256,
    ):
        super().__init__()

        # Existing language encoder
        self.student = student

        # Vision encoder
        self.vision_encoder = nn.Sequential(
            nn.Linear(vision_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.LayerNorm(latent_dim),
        )

        # Language + Vision fusion
        self.fusion = nn.Sequential(
            nn.Linear(
                latent_dim * 2,
                hidden_dim
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                latent_dim
            ),
            nn.Tanh(),
        )

        # Final action head
        self.action_head = nn.Sequential(
            nn.Linear(
                latent_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                action_dim
            )
        )

    def forward(
        self,
        tokens,
        mask,
        vision,
    ):

        # ------------------------------------
        # Language → latent
        # ------------------------------------

        student_out = self.student(
            tokens,
            mask
        )

        z_language = (
            student_out["latent"]
        )

        # ------------------------------------
        # Vision → latent
        # ------------------------------------

        z_vision = self.vision_encoder(
            vision
        )

        # ------------------------------------
        # Fusion
        # ------------------------------------

        fused = torch.cat(
            [
                z_language,
                z_vision
            ],
            dim=-1
        )

        z_fused = self.fusion(
            fused
        )

        # ------------------------------------
        # Action
        # ------------------------------------

        action = self.action_head(
            z_fused
        )

        return {
            "z_language": z_language,
            "z_vision": z_vision,
            "z_fused": z_fused,
            "action": action,
        }


In [ ]:
# ============================================
# Freeze existing Student
# ============================================

for p in student.parameters():
    p.requires_grad = False

student.eval()

vision_action_model = PGLAVisionAction(
    student=student,
    vision_dim=512,
    latent_dim=LATENT_DIM,
    action_dim=7,
).to(device)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in vision_action_model.parameters()
        if p.requires_grad
    )
)


Trainable parameters: 297991


In [ ]:
# ============================================
# Train Vision + Fusion + Action Head
# ============================================

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        vision_action_model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4,
)

EPOCHS = 20

for epoch in range(EPOCHS):

    vision_action_model.train()

    total_loss = 0.0
    n = 0

    for batch in train_loader:

        tokens = batch["tokens"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        vision = batch["vision"].to(
            device,
            non_blocking=True
        )

        action = batch["action"].to(
            device,
            non_blocking=True
        )

        output = vision_action_model(
            tokens,
            mask,
            vision,
        )

        losses = action_loss_components(
            output["action"],
            action
        )

        loss = losses["total"]

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        bs = tokens.size(0)

        total_loss += (
            loss.item() * bs
        )

        n += bs

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss {total_loss / n:.6f}"
    )


Epoch 01 | Loss 0.076437
Epoch 02 | Loss 0.057264
Epoch 03 | Loss 0.054149
Epoch 04 | Loss 0.053220
Epoch 05 | Loss 0.052968
Epoch 06 | Loss 0.052125
Epoch 07 | Loss 0.052457
Epoch 08 | Loss 0.052197
Epoch 09 | Loss 0.051769
Epoch 10 | Loss 0.051885
Epoch 11 | Loss 0.051935
Epoch 12 | Loss 0.051789
Epoch 13 | Loss 0.051474
Epoch 14 | Loss 0.051393
Epoch 15 | Loss 0.051580
Epoch 16 | Loss 0.051150
Epoch 17 | Loss 0.051197
Epoch 18 | Loss 0.051142
Epoch 19 | Loss 0.051116
Epoch 20 | Loss 0.051201


In [ ]:
# ============================================
# Evaluate Vision-conditioned Student
# ============================================

@torch.no_grad()
def evaluate_vision_student(
    model,
    loader,
):

    model.eval()

    position_errors = []
    rotation_errors = []
    grip_correct = []

    for batch in loader:

        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        vision = batch["vision"].to(device)
        target = batch["action"].to(device)

        output = model(
            tokens,
            mask,
            vision
        )

        pred = output["action"]

        position_errors.append(
            torch.abs(
                pred[:, :3]
                -
                target[:, :3]
            ).cpu()
        )

        rotation_errors.append(
            torch.abs(
                pred[:, 3:6]
                -
                target[:, 3:6]
            ).cpu()
        )

        grip_pred = (
            torch.sigmoid(
                pred[:, 6]
            ) > 0.5
        )

        grip_target = (
            target[:, 6] > 0.5
        )

        grip_correct.append(
            (
                grip_pred
                ==
                grip_target
            ).float().cpu()
        )

    position_mae = torch.cat(
        position_errors
    ).mean().item()

    rotation_mae = torch.cat(
        rotation_errors
    ).mean().item()

    gripper_accuracy = torch.cat(
        grip_correct
    ).mean().item()

    return {
        "position_mae":
            position_mae,

        "rotation_mae":
            rotation_mae,

        "gripper_accuracy":
            gripper_accuracy,
    }


In [ ]:
vision_student_results = (
    evaluate_vision_student(
        vision_action_model,
        test_loader
    )
)

print(
    json.dumps(
        vision_student_results,
        indent=2
    )
)


{
  "position_mae": 0.24049033224582672,
  "rotation_mae": 0.0056826090440154076,
  "gripper_accuracy": 1.0
}


In [ ]:
# ============================================
# Vision ablation
# ============================================

@torch.no_grad()
def evaluate_shuffled_vision(
    model,
    loader,
):

    model.eval()

    errors = []

    for batch in loader:

        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        vision = batch["vision"].to(device)
        target = batch["action"].to(device)

        # Shuffle vision across batch
        perm = torch.randperm(
            vision.size(0),
            device=device
        )

        shuffled_vision = vision[perm]

        output = model(
            tokens,
            mask,
            shuffled_vision
        )

        pred = output["action"]

        errors.append(
            torch.abs(
                pred[:, :3]
                -
                target[:, :3]
            ).cpu()
        )

    return torch.cat(
        errors
    ).mean().item()


In [ ]:
normal = vision_student_results[
    "position_mae"
]

shuffled = evaluate_shuffled_vision(
    vision_action_model,
    test_loader
)

print(
    f"Normal Vision position MAE : {normal:.4f}"
)

print(
    f"Shuffled Vision position MAE: {shuffled:.4f}"
)


Normal Vision position MAE : 0.2405
Shuffled Vision position MAE: 0.3188


In [ ]:
# ============================================
# Vision-only Action Baseline
# ============================================

class VisionOnlyAction(nn.Module):

    def __init__(
        self,
        vision_dim=512,
        action_dim=7,
        hidden_dim=256,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                vision_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                action_dim
            )
        )

    def forward(self, vision):

        return {
            "action": self.net(vision)
        }


In [ ]:
vision_only = VisionOnlyAction(
    vision_dim=512,
    action_dim=7,
).to(device)

optimizer = torch.optim.AdamW(
    vision_only.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

EPOCHS = 20

for epoch in range(EPOCHS):

    vision_only.train()

    total_loss = 0.0
    n = 0

    for batch in train_loader:

        vision = batch["vision"].to(device)
        target = batch["action"].to(device)

        output = vision_only(
            vision
        )

        losses = action_loss_components(
            output["action"],
            target
        )

        loss = losses["total"]

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()
        optimizer.step()

        bs = vision.size(0)

        total_loss += (
            loss.item() * bs
        )

        n += bs

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss {total_loss/n:.6f}"
    )


Epoch 01 | Loss 0.347480
Epoch 02 | Loss 0.344059
Epoch 03 | Loss 0.343881
Epoch 04 | Loss 0.343872
Epoch 05 | Loss 0.343607
Epoch 06 | Loss 0.343559
Epoch 07 | Loss 0.343513
Epoch 08 | Loss 0.343534
Epoch 09 | Loss 0.343288
Epoch 10 | Loss 0.343212
Epoch 11 | Loss 0.343043
Epoch 12 | Loss 0.342885
Epoch 13 | Loss 0.342967
Epoch 14 | Loss 0.342685
Epoch 15 | Loss 0.342513
Epoch 16 | Loss 0.342365
Epoch 17 | Loss 0.342163
Epoch 18 | Loss 0.341827
Epoch 19 | Loss 0.341781
Epoch 20 | Loss 0.341320


In [ ]:
# ============================================
# Evaluate Vision-only model
# ============================================

@torch.no_grad()
def evaluate_vision_only(
    model,
    loader,
):

    model.eval()

    position_errors = []
    rotation_errors = []
    grip_correct = []

    for batch in loader:

        vision = batch["vision"].to(
            device,
            non_blocking=True
        )

        target = batch["action"].to(
            device,
            non_blocking=True
        )

        output = model(
            vision
        )

        pred = output["action"]

        # ------------------------------------
        # Position
        # ------------------------------------

        position_errors.append(
            torch.abs(
                pred[:, :3]
                -
                target[:, :3]
            ).cpu()
        )

        # ------------------------------------
        # Rotation
        # ------------------------------------

        rotation_errors.append(
            torch.abs(
                pred[:, 3:6]
                -
                target[:, 3:6]
            ).cpu()
        )

        # ------------------------------------
        # Gripper
        # ------------------------------------

        grip_pred = (
            torch.sigmoid(
                pred[:, 6]
            ) > 0.5
        )

        grip_target = (
            target[:, 6] > 0.5
        )

        grip_correct.append(
            (
                grip_pred
                ==
                grip_target
            ).float().cpu()
        )

    return {
        "position_mae":
            torch.cat(
                position_errors
            ).mean().item(),

        "rotation_mae":
            torch.cat(
                rotation_errors
            ).mean().item(),

        "gripper_accuracy":
            torch.cat(
                grip_correct
            ).mean().item(),
    }


In [ ]:
vision_only_results = evaluate_vision_only(
    vision_only,
    test_loader
)

print(
    json.dumps(
        vision_only_results,
        indent=2
    )
)


{
  "position_mae": 0.2714824378490448,
  "rotation_mae": 0.0019414924317970872,
  "gripper_accuracy": 0.7512000203132629
}


In [ ]:
class LanguageOnlyAction(nn.Module):

    def __init__(
        self,
        student,
        latent_dim=128,
        action_dim=7,
        hidden_dim=256,
    ):
        super().__init__()

        self.student = student

        self.action_head = nn.Sequential(
            nn.Linear(
                latent_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                action_dim
            )
        )

    def forward(
        self,
        tokens,
        mask,
    ):

        out = self.student(
            tokens,
            mask
        )

        z = out["latent"]

        action = self.action_head(z)

        return {
            "latent": z,
            "action": action
        }


In [ ]:
language_only = LanguageOnlyAction(
    student=student,
    latent_dim=LATENT_DIM,
    action_dim=7,
).to(device)

for p in language_only.student.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        language_only.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4,
)


In [ ]:
# ============================================
# Language Baseline Encoder
# Same architecture as PGLA Student
# ============================================

class LanguageBaselineEncoder(nn.Module):

    def __init__(
        self,
        vocab_size=10000,
        embed_dim=96,
        hidden_dim=192,
        latent_dim=128,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.projector = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                256
            ),
            nn.GELU(),

            nn.Linear(
                256,
                latent_dim
            ),

            nn.Tanh()
        )

    def forward(
        self,
        tokens,
        mask=None
    ):

        x = self.embedding(tokens)

        h, _ = self.gru(x)

        if mask is None:

            pooled = h.mean(dim=1)

        else:

            mask = mask.unsqueeze(-1).float()

            pooled = (
                h * mask
            ).sum(dim=1) / mask.sum(
                dim=1
            ).clamp_min(1.0)

        z = self.projector(
            pooled
        )

        return {
            "latent": z
        }


In [ ]:
# ============================================
# Language Encoder → Action
# ============================================

class LanguageActionModel(nn.Module):

    def __init__(
        self,
        encoder,
        latent_dim=128,
        action_dim=7,
        hidden_dim=256,
    ):
        super().__init__()

        self.encoder = encoder

        self.action_head = nn.Sequential(
            nn.Linear(
                latent_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                action_dim
            )
        )

    def forward(
        self,
        tokens,
        mask,
    ):

        out = self.encoder(
            tokens,
            mask
        )

        z = out["latent"]

        action = self.action_head(
            z
        )

        return {
            "latent": z,
            "action": action
        }


In [ ]:
# ============================================
# Build baseline
# ============================================

baseline_encoder = LanguageBaselineEncoder(
    vocab_size=10000,
    embed_dim=96,
    hidden_dim=192,
    latent_dim=LATENT_DIM,
)

baseline_model = LanguageActionModel(
    encoder=baseline_encoder,
    latent_dim=LATENT_DIM,
    action_dim=7,
).to(device)

print(
    "Baseline parameters:",
    sum(
        p.numel()
        for p in baseline_model.parameters()
    )
)


Baseline parameters: 1526151


In [ ]:
# ============================================
# Train language baseline
# ============================================

optimizer = torch.optim.AdamW(
    baseline_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

EPOCHS = 20

for epoch in range(EPOCHS):

    baseline_model.train()

    total_loss = 0.0
    n = 0

    for batch in train_loader:

        tokens = batch["tokens"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        target = batch["action"].to(
            device,
            non_blocking=True
        )

        output = baseline_model(
            tokens,
            mask
        )

        losses = action_loss_components(
            output["action"],
            target
        )

        loss = losses["total"]

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        bs = tokens.size(0)

        total_loss += (
            loss.item() * bs
        )

        n += bs

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss {total_loss / n:.6f}"
    )


Epoch 01 | Loss 0.104185
Epoch 02 | Loss 0.074048
Epoch 03 | Loss 0.073767
Epoch 04 | Loss 0.073540
Epoch 05 | Loss 0.073234
Epoch 06 | Loss 0.072224
Epoch 07 | Loss 0.070100
Epoch 08 | Loss 0.069948
Epoch 09 | Loss 0.069825
Epoch 10 | Loss 0.069765
Epoch 11 | Loss 0.069754
Epoch 12 | Loss 0.069737
Epoch 13 | Loss 0.069615
Epoch 14 | Loss 0.069630
Epoch 15 | Loss 0.069633
Epoch 16 | Loss 0.069638
Epoch 17 | Loss 0.069571
Epoch 18 | Loss 0.069574
Epoch 19 | Loss 0.069700
Epoch 20 | Loss 0.069540


In [ ]:
# ============================================
# Evaluate Language Baseline
# ============================================

@torch.no_grad()
def evaluate_language_model(
    model,
    loader,
):

    model.eval()

    position_errors = []
    rotation_errors = []
    grip_correct = []

    for batch in loader:

        tokens = batch["tokens"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        target = batch["action"].to(
            device,
            non_blocking=True
        )

        output = model(
            tokens,
            mask
        )

        pred = output["action"]

        position_errors.append(
            torch.abs(
                pred[:, :3]
                -
                target[:, :3]
            ).cpu()
        )

        rotation_errors.append(
            torch.abs(
                pred[:, 3:6]
                -
                target[:, 3:6]
            ).cpu()
        )

        grip_pred = (
            torch.sigmoid(
                pred[:, 6]
            ) > 0.5
        )

        grip_target = (
            target[:, 6] > 0.5
        )

        grip_correct.append(
            (
                grip_pred
                ==
                grip_target
            ).float().cpu()
        )

    return {
        "position_mae":
            torch.cat(
                position_errors
            ).mean().item(),

        "rotation_mae":
            torch.cat(
                rotation_errors
            ).mean().item(),

        "gripper_accuracy":
            torch.cat(
                grip_correct
            ).mean().item(),
    }


In [ ]:
baseline_results = evaluate_language_model(
    baseline_model,
    test_loader
)

print(
    json.dumps(
        baseline_results,
        indent=2
    )
)


{
  "position_mae": 0.28667572140693665,
  "rotation_mae": 0.001902317046187818,
  "gripper_accuracy": 1.0
}


In [ ]:
# ============================================
# Normal Language Encoder + Vision
# Controlled Baseline
# ============================================

class BaselineLanguageVisionAction(nn.Module):

    def __init__(
        self,
        encoder,
        vision_dim=512,
        latent_dim=128,
        action_dim=7,
        hidden_dim=256,
    ):
        super().__init__()

        self.encoder = encoder

        # Vision branch
        self.vision_encoder = nn.Sequential(
            nn.Linear(
                vision_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                latent_dim
            ),

            nn.LayerNorm(
                latent_dim
            )
        )

        # Same fusion as PGLA
        self.fusion = nn.Sequential(
            nn.Linear(
                latent_dim * 2,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                latent_dim
            ),

            nn.Tanh()
        )

        # Same action head as PGLA
        self.action_head = nn.Sequential(
            nn.Linear(
                latent_dim,
                hidden_dim
            ),
            nn.GELU(),

            nn.Linear(
                hidden_dim,
                action_dim
            )
        )

    def forward(
        self,
        tokens,
        mask,
        vision,
    ):

        # Language
        language_out = self.encoder(
            tokens,
            mask
        )

        z_language = (
            language_out["latent"]
        )

        # Vision
        z_vision = self.vision_encoder(
            vision
        )

        # Fusion
        fused = torch.cat(
            [
                z_language,
                z_vision
            ],
            dim=-1
        )

        z_fused = self.fusion(
            fused
        )

        # Action
        action = self.action_head(
            z_fused
        )

        return {
            "z_language": z_language,
            "z_vision": z_vision,
            "z_fused": z_fused,
            "action": action,
        }


In [ ]:
# ============================================
# Build controlled baseline
# ============================================

baseline_lv = BaselineLanguageVisionAction(
    encoder=LanguageBaselineEncoder(
        vocab_size=10000,
        embed_dim=96,
        hidden_dim=192,
        latent_dim=LATENT_DIM,
    ),
    vision_dim=512,
    latent_dim=LATENT_DIM,
    action_dim=7,
    hidden_dim=256,
).to(device)

print(
    "Baseline L+V params:",
    sum(
        p.numel()
        for p in baseline_lv.parameters()
    )
)


Baseline L+V params: 1723527


In [ ]:
print(
    "PGLA L+V params:",
    sum(
        p.numel()
        for p in vision_action_model.parameters()
    )
)

print(
    "Baseline L+V params:",
    sum(
        p.numel()
        for p in baseline_lv.parameters()
    )
)


PGLA L+V params: 876942
Baseline L+V params: 1723527


In [ ]:
# ============================================
# Train controlled baseline
# ============================================

optimizer = torch.optim.AdamW(
    baseline_lv.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

EPOCHS = 20

for epoch in range(EPOCHS):

    baseline_lv.train()

    total_loss = 0.0
    n = 0

    for batch in train_loader:

        tokens = batch["tokens"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        vision = batch["vision"].to(
            device,
            non_blocking=True
        )

        target = batch["action"].to(
            device,
            non_blocking=True
        )

        output = baseline_lv(
            tokens,
            mask,
            vision
        )

        losses = action_loss_components(
            output["action"],
            target
        )

        loss = losses["total"]

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        bs = tokens.size(0)

        total_loss += (
            loss.item() * bs
        )

        n += bs

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss {total_loss / n:.6f}"
    )


Epoch 01 | Loss 0.091679
Epoch 02 | Loss 0.060776
Epoch 03 | Loss 0.056483
Epoch 04 | Loss 0.055972
Epoch 05 | Loss 0.055708
Epoch 06 | Loss 0.055674
Epoch 07 | Loss 0.054806
Epoch 08 | Loss 0.054862
Epoch 09 | Loss 0.054979
Epoch 10 | Loss 0.054796
Epoch 11 | Loss 0.054486
Epoch 12 | Loss 0.054566
Epoch 13 | Loss 0.054551
Epoch 14 | Loss 0.054456
Epoch 15 | Loss 0.054250
Epoch 16 | Loss 0.054291
Epoch 17 | Loss 0.054267
Epoch 18 | Loss 0.054009
Epoch 19 | Loss 0.053969
Epoch 20 | Loss 0.054217


In [ ]:
baseline_lv_results = evaluate_vision_student(
    baseline_lv,
    test_loader
)

print(
    json.dumps(
        baseline_lv_results,
        indent=2
    )
)


{
  "position_mae": 0.2457951009273529,
  "rotation_mae": 0.005489630158990622,
  "gripper_accuracy": 1.0
}


In [ ]:
import time
import torch


def benchmark_training_speed(
    model,
    loader,
    epochs=5,
    lr=1e-3,
):

    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    model.train()

    # GPU warmup
    for batch in loader:

        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        target = batch["action"].to(device)

        output = model(
            tokens,
            mask
        )

        loss = action_loss_components(
            output["action"],
            target
        )["total"]

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()
        optimizer.step()

        break

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    total_samples = 0

    for epoch in range(epochs):

        model.train()

        for batch in loader:

            tokens = batch["tokens"].to(
                device,
                non_blocking=True
            )

            mask = batch["mask"].to(
                device,
                non_blocking=True
            )

            target = batch["action"].to(
                device,
                non_blocking=True
            )

            output = model(
                tokens,
                mask
            )

            loss = action_loss_components(
                output["action"],
                target
            )["total"]

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()
            optimizer.step()

            total_samples += tokens.size(0)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = (
        time.perf_counter()
        - start
    )

    return {
        "epochs": epochs,
        "seconds": elapsed,
        "seconds_per_epoch":
            elapsed / epochs,
        "samples_per_second":
            total_samples / elapsed,
    }


In [ ]:
pgla_action_model = LanguageActionModel(
    encoder=student,
    latent_dim=LATENT_DIM,
    action_dim=7,
).to(device)


In [ ]:
baseline_action_model = baseline_model


In [ ]:
pgla_speed = benchmark_training_speed(
    pgla_action_model,
    train_loader,
    epochs=5,
)

baseline_speed = benchmark_training_speed(
    baseline_action_model,
    train_loader,
    epochs=5,
)

print("PGLA Student")
print(json.dumps(pgla_speed, indent=2))

print("\nLanguage Baseline")
print(json.dumps(baseline_speed, indent=2))


PGLA Student
{
  "epochs": 5,
  "seconds": 6.463070994999725,
  "seconds_per_epoch": 1.2926141989999451,
  "samples_per_second": 23208.781106698392
}

Language Baseline
{
  "epochs": 5,
  "seconds": 7.492231074000301,
  "seconds_per_epoch": 1.4984462148000603,
  "samples_per_second": 20020.73861823792
}


In [ ]:
torch.save(
    student.state_dict(),
    "pgla_encoder.pt"
)

torch.save(
    baseline_encoder.state_dict(),
    "language_encoder.pt"
)


In [ ]:
print(len(vocab))


15


In [ ]:
print(pgla_encoder)


LanguageBaselineEncoder(
  (embedding): Embedding(10000, 96)
  (gru): GRU(96, 192, batch_first=True, bidirectional=True)
  (projector): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): Tanh()
  )
)
